# Semantic Search: Embedding Generation Pipeline
This notebook handles the generation of dense vector representations (embeddings) for our e-commerce product catalog using various SentenceTransformer models. These embeddings will be exported and later used for vector similarity search.

## 1. Environment Setup
Install required libraries for vector encoding and data manipulation.

In [ ]:
!pip install sentence-transformers pandas numpy

## 2. Library Imports
Importing necessary libraries for data handling, timing, array operations, and NLP modeling.

In [ ]:
import pandas as pd
import time
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

# Load the sentence transformer model

# We will compare these 3 models
models_to_test = [
    'all-MiniLM-L6-v2',       # Fast, good baseline (SBERT lightweight variant)
    'all-mpnet-base-v2',      # Highly accurate, but slower and larger (SBERT advanced variant)
    'paraphrase-MiniLM-L3-v2' # Trained specifically for paraphrase identification
    ]

## 3. Load Cleaned Dataset
Load the preprocessed e-commerce dataset containing the unified `product_text` field.

In [ ]:
# Load the cleaned dataset
df = pd.read_csv("/content/drive/MyDrive/Data Science Project II/code/cleaned_flipkart_data.csv")
df

,uniq_id,image,product_name,description,brand,retail_price,discounted_price,category,product_text
0,c2d766ca982eca8304150849735ffef9,"[""http://img5a.flixcart.com/image/short/u/4/a/...",alisha solid women s cycling shorts,key features of alisha solid women s cycling s...,alisha,999.0,379.0,clothing women s clothing lingerie sleep swimw...,alisha solid women s cycling shorts key featur...
1,7f7036a6d550aaa89d34c77bd39a5e48,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",fabhomedecor fabric double sofa bed,fabhomedecor fabric double sofa bed finish col...,fabhomedecor,32157.0,22646.0,furniture living room furniture sofa beds futo...,fabhomedecor fabric double sofa bed fabhomedec...
2,f449ec65dcbc041b6ae5e6a32717d01b,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",aw bellies,key features of aw bellies sandals wedges heel...,aw,999.0,499.0,footwear women s footwear ballerinas aw bellies,aw bellies key features of aw bellies sandals ...
3,0973b37acd0c664e3de26e97e5571454,"[""http://img5a.flixcart.com/image/short/6/2/h/...",alisha solid women s cycling shorts,key features of alisha solid women s cycling s...,alisha,699.0,267.0,clothing women s clothing lingerie sleep swimw...,alisha solid women s cycling shorts key featur...
4,bc940ea42ee6bef5ac7cea3fb5cfbee7,"[""http://img5a.flixcart.com/image/pet-shampoo/...",sicons all purpose arnica dog shampoo,specifications of sicons all purpose arnica do...,sicons,220.0,210.0,pet supplies grooming skin coat care shampoo s...,sicons all purpose arnica dog shampoo specific...
...,...,...,...,...,...,...,...,...,...
19990,7179d2f6c4ad50a17d014ca1d2815156,"[""http://img6a.flixcart.com/image/wall-decorat...",walldesign small vinyl sticker,buy walldesign small vinyl sticker for rs 730 ...,walldesign,1500.0,730.0,baby care baby kids gifts stickers walldesign ...,walldesign small vinyl sticker buy walldesign ...
19991,71ac419198359d37b8fe5e3fffdfee09,"[""http://img6a.flixcart.com/image/sticker/z/g/...",wallmantra large vinyl stickers sticker,buy wallmantra large vinyl stickers sticker fo...,wallmantra,1429.0,1143.0,baby care baby kids gifts stickers wallmantra ...,wallmantra large vinyl stickers sticker buy wa...
19992,93e9d343837400ce0d7980874ece471c,"[""http://img5a.flixcart.com/image/sticker/b/s/...",elite collection medium acrylic sticker,buy elite collection medium acrylic sticker fo...,elite collection,1299.0,999.0,baby care baby kids gifts stickers elite colle...,elite collection medium acrylic sticker buy el...
19993,669e79b8fa5d9ae020841c0c97d5e935,"[""http://img5a.flixcart.com/image/sticker/4/2/...",elite collection medium acrylic sticker,buy elite collection medium acrylic sticker fo...,elite collection,1499.0,1199.0,baby care baby kids gifts stickers elite colle...,elite collection medium acrylic sticker buy el...


## 4. Prepare Corpus
Extract the unified product metadata strings into a list for batch encoding.

In [ ]:

# Prepare sentences for embedding
sentences = df['product_text'].tolist()
print(f"Number of products to encode: {len(sentences)}")

Number of products to encode: 19995


## 5. Generate and Export Embeddings
Iterate through multiple `SentenceTransformer` models, encode the entire product corpus, and save the resulting dense vectors along with the original DataFrame to disk as `.pkl` artifacts.

In [ ]:
# Loop through each model, generate embeddings, and save
for model_name in models_to_test:
    print(f"\n{'='*50}")
    print(f"Loading Model: {model_name}")
    model = SentenceTransformer(model_name)

    start_time = time.time()
    print(f"Generating embeddings for {model_name}...")
    embeddings = model.encode(sentences, show_progress_bar=True)
    encoding_time = time.time() - start_time

    print(f"Done! Shape: {embeddings.shape}. Took {encoding_time:.2f} seconds.")

    # Save embeddings
    output_file = f"embeddings_{model_name.replace('/', '_')}.pkl"
    with open(output_file, 'wb') as f:
        pickle.dump({'embeddings': embeddings, 'df': df, 'model_name': model_name}, f)

    print(f"Saved to {output_file}")


Loading Model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for paraphrase-MiniLM-L3-v2...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Done! Shape: (19995, 384). Took 18.43 seconds.
Saved to embeddings_paraphrase-MiniLM-L3-v2.pkl
